# Field Theory Framework — Demo

Model-agnostic MSRJD mean-field expansion framework.  
Demonstrated here on the **nonlinear Hawkes 2-population** model.

**Kernel**: Python 3 (SymPy).  
SageMath extension points are noted in comments where applicable.

### What this notebook does
1. Load the `FieldTheory` class and `HAWKES_MODEL` spec dict
2. Build the symbolic namespace (fields, parameters, operators)
3. Expand and classify the action by bigrade $(n_{\tilde}, n_{\text{phys}})$
4. Run sanity checks (tadpole cancellation, constant term, EOM residual)
5. Display each sector: free action, noise kernel, vertices
6. Write the free action in explicit Conv/IP operator notation

In [ ]:
import sys, os
# Ensure the Framework directory is on the path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import sympy as sp
sp.init_printing(use_latex='mathjax')

from field_theory import FieldTheory, Conv, IP
from models.hawkes import HAWKES_MODEL

print('Loaded model:', HAWKES_MODEL['name'])
print('Background convention:', HAWKES_MODEL['background_rate_convention'])

## 1. Build and expand

Instantiate `FieldTheory`, call `expand()` to:
- construct the symbolic namespace from the model dict
- evaluate the action lambda
- classify all terms by bigrade $(n_{\tilde}, n_{\text{phys}})$

In [ ]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()
print('Expansion complete.')
print('Non-zero bigrade sectors found:', sorted(ft.sectors().keys()))

## 2. Sanity checks

Three sectors must vanish:
- $(0,0)$ — constant (background energy, cancels by construction)
- $(1,0)$ — tadpole: linear in $\tilde{n}_i, \tilde{v}_i$ only; vanishes at MF saddle $\dot{n}_i^* = -\varphi_i(v_i^*)$
- $(0,1)$ — EOM residual: linear in physical fluctuations only; vanishes at background EOM

In [ ]:
passed = ft.sanity_check()

## 3. Full sector summary

In [ ]:
ft.summary()

## 4. Free action  $(n_{\tilde}=1,\, n_{\text{phys}}=1)$

The free (quadratic) action — determines propagators and the Gaussian integral kernel.

In [ ]:
from IPython.display import display, Math

S_free = ft.free_action()
print('Free action S_free (expanded):')
display(S_free)
print('\nLaTeX:')
print(sp.latex(S_free))

### Structural check: every term in free action must be exactly $(1,1)$

In [ ]:
ns = ft._ns
tilde_syms = ns._tilde_syms
phys_syms  = ns._phys_syms

all_ok = True
for term in sp.Add.make_args(sp.expand(S_free)):
    n_t = sum(int(sp.degree(term, x)) for x in tilde_syms)
    n_p = sum(int(sp.degree(term, x)) for x in phys_syms)
    if (n_t, n_p) != (1, 1):
        print(f'  [FAIL] unexpected (n_tilde={n_t}, n_phys={n_p}): {term}')
        all_ok = False
if all_ok:
    print('[PASS] All free-action terms have bigrade (1,1).')

## 5. Noise kernel  $(n_{\tilde} \geq 2,\, n_{\text{phys}}=0)$

Source terms quadratic (and higher) in response fields — generate propagator corrections.

In [ ]:
nk = ft.noise_kernel()
print(f'Noise kernel sectors: {sorted(nk.keys())}')
for (n_t, n_p), expr in sorted(nk.items()):
    print(f'\n  (n_tilde={n_t}, n_phys={n_p}):')
    display(expr)

## 6. Vertices  (total degree $\geq 3$)

Interaction vertices — enter as Feynman diagram insertions.

In [ ]:
verts = ft.vertices()
print(f'Vertex sectors: {sorted(verts.keys())}')
for (n_t, n_p), expr in sorted(verts.items()):
    print(f'\n  (n_tilde={n_t}, n_phys={n_p}):')
    display(expr)

## 7. Free action in Conv / IP operator notation

Every bilinear coupling is written as $\mathrm{IP}(\tilde{\phi}, \mathrm{Conv}(\kappa, \psi))$
with explicit convolution kernels:

| Operator | Kernel $\kappa$ |
|---|---|
| identity | $\delta(t)$ |
| gain | $\varphi_i'\,\delta(t)$ |
| synaptic filter | $g(t)$ |
| voltage operator $(\tau\partial_t + 1)$ | $\tau\delta'(t) + \delta(t)$ |

In [ ]:
delta_D  = ns.delta_D    # δ(t)
delta_Dp = ns.delta_Dp   # δ'(t)
tau      = ns.tau
pop      = ns.pop
nt_list  = ns.nt
vt_list  = ns.vt
dn_list  = ns.dn
dv_list  = ns.dv
w        = ns.w
g        = ns.g
phi1s    = [sp.symbols(f'phi1_{i+1}') for i in pop]

S_free_conv = sp.Integer(0)
for i in pop:
    # ñ_i  * Conv(δ, δṅ_i)  — spike-train coupling
    S_free_conv += IP(nt_list[i],  Conv(delta_D,                    dn_list[i]))
    # ñ_i  * Conv(φ'_i δ, δv_i)  — gain coupling
    S_free_conv += IP(nt_list[i],  Conv(phi1s[i] * delta_D,         dv_list[i]))
    # ṽ_i  * Conv(τδ' + δ, δv_i)  — voltage operator
    S_free_conv += IP(vt_list[i],  Conv(tau * delta_Dp + delta_D,   dv_list[i]))
    for j in pop:
        # -ṽ_i * Conv(w_{ij} g, δṅ_j)  — synaptic coupling
        S_free_conv -= IP(vt_list[i], Conv(w[i][j] * g,             dn_list[j]))

print('Free action in Conv/IP notation:')
display(S_free_conv)
print('\nLaTeX:')
print(sp.latex(S_free_conv))

## 8. Inspect raw expanded action (optional)

View all terms in the raw expanded action $S$ before MF substitution.

In [ ]:
print('Raw expanded action S:')
display(ft._S_raw)

---
## Using a different model

To apply the framework to any other stochastic system:

```python
MY_MODEL = {
    'name': 'My model',
    'index_sets':      {'pop': [0]},
    'physical_fields': [{'name': 'dn', 'indexed': True}],
    'response_fields': [{'name': 'nt', 'indexed': True}],
    'parameters':      [{'name': 'mu', 'indexed': False}],
    'kernels':         [],
    'operators':       [],
    'functions':       [...],        # Taylor expansions
    'mf_substitutions': [...],       # background values
    'background_rate_convention': '...',
    'action': lambda ns: ...         # full symbolic action
}

ft2 = FieldTheory(MY_MODEL, taylor_order=4)
ft2.expand()
ft2.sanity_check()
ft2.summary()
```

See `models/hawkes.py` for the full format reference.